# FBSA — Evaluation & Visualization

Pick an arch via `ARCH` (top of cell-1):
- `fbsa`        — v1 (slot only)
- `fbsa_fused`  — v2 (option 1: slot + encoder fusion)
- `fbsa_skip`   — v3 (option 2: + image-stem multi-scale skips)

Loads `best_model.pt` from that arch's run dir, computes test metrics, and shows
`image | gt | pred | TP/FP/FN overlay` per sample. The overlay is what reveals
boundary quality — Dice can be the same while one model has clean edges and the
other has ragged ones.

The final section loads UNet + DINO baseline + your selected FBSA variant and
renders a unified 5-column grid: `image | gt | unet | dino | fbsa_skip`, each
model shown as a TP/FP/FN overlay so boundary differences are obvious.

In [ ]:
import sys, random
from pathlib import Path
PROJECT = Path('..').resolve()
sys.path.insert(0, str(PROJECT))
import numpy as np
from PIL import Image
import torch, torch.nn.functional as F, kagglehub, matplotlib.pyplot as plt
import torchvision.transforms.functional as TF
from tumor_seg.config import TrainConfig
from tumor_seg.data import create_brisc_dataloaders
from tumor_seg.metrics import dice_coefficient, iou_coefficient, hausdorff_distance_metric
from tumor_seg.models import build_model, ARCH_REGISTRY
from tumor_seg.baselines import UNet, DinoViTSegmentationModel, build_dino_baseline

ARCH = 'fbsa_skip'                                            # ← v3
FBSA_RUN_DIR     = f'/content/drive/MyDrive/fbsa_runs/{ARCH}'
UNET_CKPT_PATH   = '/content/drive/MyDrive/baseline_runs/unet/best_model.pt'
DINO_CKPT_PATH   = '/content/drive/MyDrive/baseline_runs/dino_convtranspose/best_model.pt'
ckpt_path = f'{FBSA_RUN_DIR}/best_model.pt'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
cfg = TrainConfig(**ckpt['cfg'])
model = build_model(cfg).to(device)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'arch = {cfg.arch}  best epoch = {ckpt["epoch"]}')

In [ ]:
data_root = str(Path(kagglehub.dataset_download('briscdataset/brisc2025')) / 'brisc2025' / 'segmentation_task')
_, val_loader = create_brisc_dataloaders(data_root, batch_size=cfg.batch_size, image_size=cfg.image_size)

tot_d = tot_i = tot_h = 0.0; n = 0
all_imgs, all_gts, all_preds = [], [], []
with torch.no_grad():
    for images, masks in val_loader:
        images, masks = images.to(device), masks.to(device)
        logits = model(images)
        bs = images.size(0); n += bs
        tot_d += dice_coefficient(logits, masks).item() * bs
        tot_i += iou_coefficient(logits, masks).item() * bs
        tot_h += hausdorff_distance_metric(logits, masks, image_size=cfg.image_size) * bs
        all_imgs.append(images.cpu()); all_gts.append(masks.cpu())
        all_preds.append((torch.sigmoid(logits) > 0.5).float().cpu())
print(f'[{cfg.arch}]  Dice = {tot_d/n:.4f}  IoU = {tot_i/n:.4f}  HD = {tot_h/n:.2f} px')

In [ ]:
# Per-image visualization for the selected FBSA arch only.
# image | gt | pred | TP/FP/FN overlay
# Overlay: green=TP, red=FP (false alarm), blue=FN (missed tumor).
imgs = torch.cat(all_imgs); gts = torch.cat(all_gts); preds = torch.cat(all_preds)

def make_error_overlay(image_chw, gt_hw, pred_hw):
    img = image_chw.permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    base = (img.mean(axis=2, keepdims=True).repeat(3, axis=2)) * 0.6
    gt = gt_hw.numpy() > 0.5
    pr = pred_hw.numpy() > 0.5
    overlay = base.copy()
    overlay[gt &  pr] = [0.1, 0.9, 0.1]
    overlay[~gt &  pr] = [0.95, 0.2, 0.2]
    overlay[gt & ~pr] = [0.2, 0.4, 0.95]
    return img, overlay

k = min(10, imgs.shape[0])
random.seed(0)
idxs = random.sample(range(imgs.shape[0]), k)
fig, axes = plt.subplots(k, 4, figsize=(12, 3 * k))
if k == 1:
    axes = np.expand_dims(axes, 0)
for r, i in enumerate(idxs):
    img, overlay = make_error_overlay(imgs[i], gts[i, 0], preds[i, 0])
    axes[r, 0].imshow(img);                         axes[r, 0].set_title('image')
    axes[r, 1].imshow(gts[i, 0], cmap='gray');      axes[r, 1].set_title('gt')
    axes[r, 2].imshow(preds[i, 0], cmap='gray');    axes[r, 2].set_title('pred')
    axes[r, 3].imshow(overlay);                     axes[r, 3].set_title('TP / FP / FN')
    for j in range(4):
        axes[r, j].axis('off')
plt.suptitle(f'{cfg.arch}  (epoch {ckpt["epoch"]})', y=1.001)
plt.tight_layout(); plt.show()

## Unified comparison: UNet vs DINO baseline vs FBSA

Loads all three checkpoints and renders one row per sample:
`image | gt | unet | dino | fbsa_skip`. Each model is shown as a TP/FP/FN
overlay so boundary differences are obvious — Dice alone won't tell you
whether v3's gain came from cleaner edges or larger blobs.

Each model has its own preprocessing:
- **UNet**: 1-channel grayscale, `[0,1]` (no ImageNet norm)
- **DINO baseline**: 3-channel RGB, `[0,1]` (no ImageNet norm — matches the notebook's loader)
- **FBSA**: 3-channel RGB, ImageNet mean/std (matches `tumor_seg.data.brisc`)

We re-load each test image from disk so each model gets its own clean preprocessing path.

In [ ]:
# --- Load UNet ---
unet_ckpt = torch.load(UNET_CKPT_PATH, map_location=device, weights_only=False)
unet = UNet(in_channels=unet_ckpt.get('in_channels', 1), out_channels=1).to(device)
unet.load_state_dict(unet_ckpt['model']); unet.eval()
print(f'UNet  loaded (epoch {unet_ckpt["epoch"]})')

# --- Load DINO baseline ---
dino_ckpt = torch.load(DINO_CKPT_PATH, map_location=device, weights_only=False)
dino_baseline = build_dino_baseline(image_size=dino_ckpt.get('image_size', 224), out_channels=1).to(device)
dino_baseline.load_state_dict(dino_ckpt['model']); dino_baseline.eval()
print(f'DINO  loaded (epoch {dino_ckpt["epoch"]})')

# FBSA model already loaded above as `model` from cell-1.
print(f'FBSA  loaded (epoch {ckpt["epoch"]})  arch={cfg.arch}')

In [ ]:
# Per-arch preprocessing. Each takes a PIL RGB image (already resized to 224)
# and returns a tensor on `device` ready for the corresponding model.
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
IMG_SIZE = cfg.image_size

def _to_resized_pil(image_path):
    return Image.open(image_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)

def preprocess_unet(pil_rgb):
    arr = np.asarray(pil_rgb.convert('L'), dtype=np.float32) / 255.0  # 1ch grayscale
    return torch.from_numpy(arr[None, None, ...]).to(device)

def preprocess_dino_baseline(pil_rgb):
    arr = np.asarray(pil_rgb, dtype=np.float32) / 255.0                 # /255 RGB
    return torch.from_numpy(arr.transpose(2, 0, 1)[None, ...]).to(device)

def preprocess_fbsa(pil_rgb):
    arr = np.asarray(pil_rgb, dtype=np.float32) / 255.0
    t = torch.from_numpy(arr.transpose(2, 0, 1)).to(device)
    t = TF.normalize(t, IMAGENET_MEAN, IMAGENET_STD)
    return t.unsqueeze(0)

MODELS = [
    ('unet',      unet,          preprocess_unet),
    ('dino',      dino_baseline, preprocess_dino_baseline),
    (cfg.arch,    model,         preprocess_fbsa),
]

@torch.no_grad()
def predict_binary(m, x):
    return (torch.sigmoid(m(x)) > 0.5).float().cpu().numpy()[0, 0]

def load_gt(mask_path, size=IMG_SIZE):
    pil_mask = Image.open(mask_path).convert('L').resize((size, size), Image.Resampling.NEAREST)
    return (np.asarray(pil_mask, dtype=np.float32) > 0).astype(np.float32)

In [ ]:
# Build the unified grid by re-reading raw images from disk.
from tumor_seg.data import BriscSegmentationDataset
raw_ds = BriscSegmentationDataset(data_root, split='test', image_size=IMG_SIZE,
                                   augment=False, rgb_input=True, normalize=False)
samples = raw_ds.samples

K = min(10, len(samples))
random.seed(0)
sample_idxs = random.sample(range(len(samples)), K)

ncols = 2 + len(MODELS)
fig, axes = plt.subplots(K, ncols, figsize=(3 * ncols, 3 * K))
if K == 1:
    axes = np.expand_dims(axes, 0)

for row, idx in enumerate(sample_idxs):
    image_path, mask_path = samples[idx]
    pil_rgb = _to_resized_pil(image_path)
    img_disp = np.asarray(pil_rgb, dtype=np.float32) / 255.0
    gt = load_gt(mask_path)

    axes[row, 0].imshow(img_disp);                axes[row, 0].set_title('image')
    axes[row, 1].imshow(gt, cmap='gray');         axes[row, 1].set_title('gt')

    for c, (name, m, prep) in enumerate(MODELS):
        x = prep(pil_rgb)
        pr = predict_binary(m, x)
        # TP/FP/FN overlay over a desaturated copy of the original image
        gray = img_disp.mean(axis=2, keepdims=True).repeat(3, axis=2) * 0.6
        overlay = gray.copy()
        gtb = gt > 0.5
        prb = pr > 0.5
        overlay[gtb &  prb] = [0.1, 0.9, 0.1]
        overlay[~gtb &  prb] = [0.95, 0.2, 0.2]
        overlay[gtb & ~prb] = [0.2, 0.4, 0.95]
        axes[row, 2 + c].imshow(overlay)
        axes[row, 2 + c].set_title(name)

    for j in range(ncols):
        axes[row, j].axis('off')

plt.suptitle('TP (green)  /  FP (red)  /  FN (blue)', y=1.001)
plt.tight_layout(); plt.show()

In [ ]:
# Aggregate metrics across the full test set, all three models, same images.
all_metrics = {name: {'dice': 0.0, 'iou': 0.0, 'hd': 0.0, 'n': 0} for name, _, _ in MODELS}

for image_path, mask_path in samples:
    pil_rgb = _to_resized_pil(image_path)
    gt_np = load_gt(mask_path)
    gt_t = torch.from_numpy(gt_np[None, None, ...]).float().to(device)
    for name, m, prep in MODELS:
        x = prep(pil_rgb)
        with torch.no_grad():
            logits = m(x)
        all_metrics[name]['dice'] += dice_coefficient(logits, gt_t).item()
        all_metrics[name]['iou']  += iou_coefficient(logits, gt_t).item()
        all_metrics[name]['hd']   += hausdorff_distance_metric(logits, gt_t, image_size=IMG_SIZE)
        all_metrics[name]['n']    += 1

print(f'{"arch":<14}{"Dice":>10}{"IoU":>10}{"HD (px)":>12}')
for name, s in all_metrics.items():
    n = max(s['n'], 1)
    print(f'{name:<14}{s["dice"]/n:>10.4f}{s["iou"]/n:>10.4f}{s["hd"]/n:>12.2f}')